# Citco Document Intelligence Workshop — Evaluation

Interactive, component-by-component evaluation of the deployed workshop. Run **after** `00_setup`. Each section produces the metrics tables the app's insights tab surfaces (all in `cdi_eval`). MLflow uses the **workspace** store (never local sqlite).

1. **setup**
2. **Engine** — parse / chunk / extract performance, scale projection, robustness (→ `engine_perf`)
3. **RAG quality** — head-to-head across arms (KA / index / supervisor / Lakebase) with MLflow GenAI judges (→ MLflow runs → `benchmark_scores`)
4. **Edge cases** — amendment precedence, table stitch, multimodal figure, MFN ladder vs answer keys (→ `edge_case_scores`)
5. **Cross-cutting QA** — the curated hybrid golden set (→ `cross_cutting_qa`; feeds the app's suggested questions)
6. **Summary rollup** — `benchmark_scores` + `pipeline_metrics` for the app's insights tab

In [ ]:
# §1 — setup: repo importable, ambient (job) auth, MLflow -> workspace store.
import os, sys
sys.path.insert(0, os.path.abspath(".."))
os.environ["CDI_USE_DEFAULT_AUTH"] = "1"

import cdi_common as C
import cdi_schemas as S
w = C.ws()

import mlflow
mlflow.set_tracking_uri("databricks")            # workspace MLflow, never local sqlite
mlflow.set_registry_uri("databricks-uc")
EXPERIMENT = os.environ.get("CDI_EVAL_EXPERIMENT", "/Shared/citco-cdi/rag-benchmark")
# mlflow.set_experiment won't create missing parent workspace dirs — ensure them first.
w.workspace.mkdirs(EXPERIMENT.rsplit("/", 1)[0])
mlflow.set_experiment(EXPERIMENT)
print("eval experiment:", EXPERIMENT, "| catalog:", C.CATALOG, "| lakebase arm:", C.LAKEBASE_SEARCH_ENABLED)

## §2 — Engine (performance · scale · robustness)

Per-stage wall clocks for `ai_parse_document` / `ai_prep_search` / `ai_extract` across small / medium / large docs, a throughput projection, and the robustness gate (the 5 messy edge files each reach exactly one terminal state — parsed XOR dead-lettered). Writes `cdi_eval.engine_perf`.

In [ ]:
# §2.0 — engine-benchmark config: SDP tables, scratch tables, parse knobs, measured M3 facts
from cdi_schemas import SL_SCHEMA, CA_SCHEMA, SL_INSTR, CA_INSTR, _sql_str
import time, os

CAT, SILVER, GOLD, EVAL, BRONZE = C.CATALOG, C.SCHEMA_SILVER, C.SCHEMA_GOLD, C.SCHEMA_EVAL, C.SCHEMA_BRONZE
MANIFEST   = f"{CAT}.{BRONZE}.corpus_manifest"
PARSED     = f"{CAT}.{SILVER}.parsed"          # SDP silver doc-text unit (VARIANT)
DEAD       = f"{CAT}.{SILVER}._dead_letter"    # honest-failure sidecar
DOC_CHUNKS = f"{CAT}.{SILVER}.doc_chunks"
SC_PARSE   = f"{CAT}.{EVAL}._bench_scratch_parse"   # raw ai_parse_document VARIANT (scratch)
SC_CHUNK   = f"{CAT}.{EVAL}._bench_scratch_chunks"  # ai_prep_search chunks for one doc (scratch)
SC_FACTS   = f"{CAT}.{EVAL}._bench_scratch_facts"   # per-doc ai_extract output (scratch)
PERF       = f"{CAT}.{EVAL}.engine_perf"

PARSE_VERSION = os.environ.get("CDI_PARSE_VERSION", "2.0")
PAGE_LIMIT    = int(os.environ.get("CDI_PARSE_PAGE_LIMIT", 500))
POLL_S        = int(os.environ.get("CDI_BENCH_POLL_S", 900))   # large-doc parse is slow
M3_EXTRACT_SECONDS = float(os.environ.get("CDI_M3_EXTRACT_SECONDS", 149.0))  # M3 measured
M3_EXTRACT_DOCS    = int(os.environ.get("CDI_M3_EXTRACT_DOCS", 100))
SCHEMAS = {"side_letter": (SL_SCHEMA, SL_INSTR), "credit_agreement": (CA_SCHEMA, CA_INSTR)}

def _esc(s): return s.replace("'", "''")
def _timed(sql, poll_s=POLL_S):
    t0 = time.time(); C.q(w, sql, poll_s=poll_s); return time.time() - t0

In [ ]:
# §2.1 — create engine_perf (CREATE OR REPLACE — eval table, nothing depends on it)
C.q(w, f"""CREATE OR REPLACE TABLE {PERF} (
             stage STRING, doc_bucket STRING, doc_path STRING, doc_type STRING,
             n_pages INT, n_elements INT, n_sections INT, n_segments INT,
             seconds DOUBLE, throughput_note STRING, measured_at TIMESTAMP) USING DELTA""")

def perf_row(stage, bucket, path, doc_type, n_pages, n_elements, n_sections, n_segments, seconds, note):
    fn = path.rsplit("/", 1)[-1]
    C.q(w, f"""INSERT INTO {PERF} VALUES
        ('{stage}', '{bucket}', '{_esc(fn)}', '{_esc(doc_type or "")}',
         {n_pages if n_pages is not None else 'NULL'},
         {n_elements if n_elements is not None else 'NULL'},
         {n_sections if n_sections is not None else 'NULL'},
         {n_segments if n_segments is not None else 'NULL'},
         {seconds:.3f}, '{_esc(note)}', current_timestamp())""")

In [ ]:
# §2.2 — pick one representative doc per size bucket (resolved LIVE from the manifest)
def _row(r):
    return {"path": r[0], "doc_type": r[1],
            "page_count": int(r[2]) if r[2] not in (None, "") else None,
            "size_bytes": int(r[3]) if r[3] not in (None, "") else None}

docs = []
for dt in ("side_letter", "credit_agreement"):  # SMALL: smallest synthetic SL + CA
    r = C.rows(C.q(w, f"""SELECT path, doc_type, page_count, size_bytes FROM {MANIFEST}
                          WHERE source='synthetic' AND doc_type='{dt}'
                          ORDER BY size_bytes ASC LIMIT 1"""))
    if r: docs.append({"bucket": "small", **_row(r[0])})
r = C.rows(C.q(w, f"""SELECT path, doc_type, page_count, size_bytes FROM {MANIFEST}  -- MEDIUM: real CA ~100-280pp
                      WHERE source='real' AND doc_type='credit_agreement'
                        AND page_count BETWEEN 100 AND 280 ORDER BY page_count DESC LIMIT 1"""))
if r: docs.append({"bucket": "medium", **_row(r[0])})
r = C.rows(C.q(w, f"""SELECT path, doc_type, page_count, size_bytes FROM {MANIFEST}  -- LARGE: messy_over500pp.pdf (~603pp)
                      WHERE path LIKE '%messy_over500pp.pdf' LIMIT 1"""))
if r: docs.append({"bucket": "large", **_row(r[0])})
docs

In [ ]:
# §2.3 — stage (a): time ai_parse_document into SC_PARSE (read-only on the PDF; segment >500pp like the SDP split)
def page_ranges(page_count):
    out, start = [], 1
    while start <= page_count:
        end = min(start + PAGE_LIMIT - 1, page_count); out.append(f"{start}-{end}"); start = end + 1
    return out

def parse_map(rng):
    return f"map('version','{PARSE_VERSION}','pageRange','{rng}')" if rng else f"map('version','{PARSE_VERSION}')"

def time_parse(path, page_count):
    """Persist one parse VARIANT row per segment -> stages (b)/(c) chunk+extract off the SAME parse."""
    p = _esc(path)
    segmented = page_count is not None and page_count > PAGE_LIMIT
    ranges = page_ranges(page_count) if segmented else [None]
    C.q(w, f"""CREATE TABLE IF NOT EXISTS {SC_PARSE}
                 (path STRING, segment STRING, page_lo INT, parsed VARIANT) USING DELTA""")
    C.q(w, f"DELETE FROM {SC_PARSE} WHERE path = '{p}'")
    total_sec = total_el = total_pg = 0
    for i, rng in enumerate(ranges):
        seg = rng or "full"; page_lo = (i * PAGE_LIMIT) + 1
        total_sec += _timed(f"""
            INSERT INTO {SC_PARSE}
            SELECT '{p}', '{_esc(seg)}', {page_lo}, ai_parse_document(content, {parse_map(rng)})
            FROM read_files('{p}', format => 'binaryFile')""")
        r = C.rows(C.q(w, f"""
            SELECT size(try_cast(parsed:document:elements AS ARRAY<VARIANT>)),
                   size(try_cast(parsed:document:pages    AS ARRAY<VARIANT>))
            FROM {SC_PARSE} WHERE path = '{p}' AND segment = '{_esc(seg)}'"""))[0]
        total_el += int(r[0] or 0); total_pg += int(r[1] or 0)
    return total_sec, total_el, total_pg, len(ranges)

In [ ]:
# §2.4 — stage (b): time managed chunking via ai_prep_search (the SDP doc_chunks SQL, one doc, over SC_PARSE)
def time_chunk(path):
    p = _esc(path)
    sec = _timed(f"""
    CREATE OR REPLACE TABLE {SC_CHUNK} AS
    WITH prepped AS (
      SELECT path, page_lo, ai_prep_search(parsed) AS result FROM {SC_PARSE}
      WHERE path = '{p}' AND parsed IS NOT NULL
        AND size(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) > 0
    ),
    exploded AS (
      SELECT (coalesce(prepped.page_lo, 1) - 1)
               + coalesce(chunk.value:pages[0].page_id::INT, 0) + 1 AS page,
             chunk.value:chunk_to_embed::STRING AS chunk_to_embed
      FROM prepped, LATERAL variant_explode(prepped.result:document.contents) AS chunk
    )
    SELECT md5(concat_ws('||', '{p}', chunk_to_embed)) AS chunk_id, page,
           chunk_to_embed AS chunk_text, length(chunk_to_embed) AS char_len
    FROM exploded
    WHERE chunk_to_embed IS NOT NULL AND length(trim(chunk_to_embed)) >= 15""")
    n = int(C.rows(C.q(w, f"SELECT count(*) FROM {SC_CHUNK}"))[0][0])
    return sec, n

In [ ]:
# §2.5 — stage (c): time one set-based ai_extract over the reassembled element text of this doc
def time_extract(path, doc_type):
    if doc_type not in SCHEMAS:                       # ai_extract only defined for SL / CA
        return 0.0, 0
    schema, instr = SCHEMAS[doc_type]
    schema_s, instr_s, p = _sql_str(schema), instr.replace("'", "''"), _esc(path)
    sec = _timed(f"""
        CREATE OR REPLACE TABLE {SC_FACTS} AS
        WITH elements AS (
          SELECT coalesce(page_lo, 0) AS seg_lo, CAST(el:bbox[0]:page_id AS INT) AS pg,
                 CAST(el:id AS BIGINT) AS eid, el:content::string AS content
          FROM {SC_PARSE}
          LATERAL VIEW explode(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS el
          WHERE path = '{p}' AND parsed IS NOT NULL
        ),
        doc AS (
          SELECT array_join(transform(
                   array_sort(collect_list(struct(seg_lo, pg, eid, content))),
                   s -> s.content), ' ') AS doc_text
          FROM elements
        )
        SELECT doc_text, ai_extract(doc_text, '{schema_s}', map('instructions', '{instr_s}')) AS ext
        FROM doc WHERE length(doc_text) > 0""")
    n = int(C.rows(C.q(w, f"SELECT count(*) FROM {SC_FACTS}"))[0][0])
    return sec, n

In [ ]:
# §2.6 — throughput projection (derived from the M3 measured set-based extract fact; methodology stated)
n_parsed = int(C.rows(C.q(w, f"""SELECT count(distinct coalesce(parent_path, path)) FROM {PARSED}
                                 WHERE parsed IS NOT NULL AND n_elements > 0"""))[0][0])
pages    = int(C.rows(C.q(w, f"SELECT coalesce(sum(n_pages),0) FROM {PARSED}"))[0][0])
chunks   = int(C.rows(C.q(w, f"SELECT count(*) FROM {DOC_CHUNKS}"))[0][0])
docs_per_hr = M3_EXTRACT_DOCS / M3_EXTRACT_SECONDS * 3600.0
methodology = (f"Set-based ai_extract (M3 measured): {M3_EXTRACT_DOCS} docs in {M3_EXTRACT_SECONDS:.0f}s "
               f"wall clock = {docs_per_hr:,.0f} docs/hr extracted. One distributed SQL statement per "
               f"doc_type over ALL docs — the FM Batch-Inference service parallelizes; per-doc latency is "
               f"NOT additive. SDP corpus today: {n_parsed} parsed docs / {pages} pages / {chunks} chunks.")
projection = {"set_based_extract_docs_per_hour": round(docs_per_hr),
              "m3_extract_docs": M3_EXTRACT_DOCS, "m3_extract_seconds": M3_EXTRACT_SECONDS,
              "corpus_parsed_docs": n_parsed, "corpus_pages": pages, "corpus_chunks": chunks}
print(methodology); projection

In [ ]:
# §2.7 — run the per-stage benchmark over the 3 bucket docs (live wall clocks) + persist to engine_perf
import json
results = []
for d in docs:
    fn = d["path"].rsplit("/", 1)[-1]
    print(f"\n[{d['bucket']}] {fn} (doc_type={d['doc_type']}, pages={d['page_count']})", flush=True)

    p_sec, n_el, n_pg, n_seg = time_parse(d["path"], d["page_count"])
    print(f"  (a) ai_parse_document:       {p_sec:6.1f}s  -> {n_el} elems / {n_pg} pages / {n_seg} seg(s)")
    perf_row("parse", d["bucket"], d["path"], d["doc_type"], n_pg, n_el, None, n_seg,
             p_sec, methodology if d["bucket"] == "small" else "")

    c_sec, n_chunk = time_chunk(d["path"])
    print(f"  (b) ai_prep_search chunking: {c_sec:6.1f}s  -> {n_chunk} chunks")
    perf_row("chunk", d["bucket"], d["path"], d["doc_type"], n_pg, n_el, n_chunk, None,
             c_sec, f"{n_chunk} managed chunks")

    x_sec, n_xdoc = time_extract(d["path"], d["doc_type"])
    if n_xdoc:
        print(f"  (c) set-based ai_extract:    {x_sec:6.1f}s  -> {n_xdoc} doc(s) extracted")
        perf_row("extract", d["bucket"], d["path"], d["doc_type"], n_pg, n_el, n_xdoc, None,
                 x_sec, "set-based ai_extract over reassembled doc text")
    else:
        print(f"  (c) set-based ai_extract:    skipped (doc_type={d['doc_type']} has no extract schema)")

    results.append({"bucket": d["bucket"], "file": fn, "doc_type": d["doc_type"], "pages": n_pg,
                    "elements": n_el, "segments": n_seg, "chunks": n_chunk,
                    "parse_s": round(p_sec, 1), "chunk_s": round(c_sec, 2), "extract_s": round(x_sec, 1)})

perf_row("throughput", "all", "(corpus)", "", projection["corpus_pages"], None, None, None,
         M3_EXTRACT_SECONDS, methodology)
print("\n=== PER-STAGE LATENCY BY BUCKET ===\n" + json.dumps(results, indent=2, default=str))

In [ ]:
# §2.8 — robustness gate: the messy edge files each reach EXACTLY ONE terminal state (parsed XOR dead_letter)
MESSY = [
    ("messy_over50mb.pdf",              "over100mb (byte-split -> parsed segments)"),
    ("messy_over500pp.pdf",             "over500pp (segmented parse)"),
    ("messy_with_attachment.pdf",       "embedded-attachment (CSV extracted + parent parsed)"),
    ("messy_scanned_ocr_only.pdf",      "scanned-OCR (recovered)"),
    ("messy_corrupt_recoverable.pdf",   "corrupt-recoverable (parsed)"),
    ("messy_corrupt_unrecoverable.pdf", "corrupt-unrecoverable (honest dead_letter)"),
]
rob = {}
for fn, expected in MESSY:
    like = f"%{fn}"
    parsed = C.rows(C.q(w, f"""
        SELECT count(*), coalesce(sum(n_pages),0), coalesce(sum(n_elements),0), count(distinct segment)
        FROM {PARSED}
        WHERE (path LIKE '{like}' OR parent_path LIKE '{like}') AND parsed IS NOT NULL
          AND (error_status IS NULL OR error_status IN ('null','')) AND n_elements > 0"""))
    dead = C.rows(C.q(w, f"""SELECT stage, substr(reason,1,160) FROM {DEAD}
        WHERE path LIKE '{like}' OR parent_path LIKE '{like}' ORDER BY failed_at DESC LIMIT 1"""))
    n_parsed = int(parsed[0][0] or 0); is_parsed = n_parsed > 0; is_dead = len(dead) > 0
    rob[fn] = {
        "expected": expected,
        "terminal": "parsed" if is_parsed else ("dead_letter" if is_dead else "MISSING"),
        "exactly_one": is_parsed != is_dead,   # parsed XOR dead-lettered
        "parsed": {"n_pages": int(parsed[0][1] or 0), "n_elements": int(parsed[0][2] or 0)} if is_parsed else None,
        "dead_letter": {"stage": dead[0][0], "reason": dead[0][1]} if is_dead else None,
        "n_segments": int(parsed[0][3] or 0),
    }
all_one = all(v["exactly_one"] for v in rob.values())
print(json.dumps(rob, indent=2, default=str))
print(f"\nRESULT: {'CDI3_BENCH_PASS' if all_one else 'CDI3_BENCH_FAIL'} "
      f"({len(MESSY)} messy = exactly one terminal state: {all_one})")

# §2.8b — embedded-attachment extraction (req #4): the CSV embedded in messy_with_attachment.pdf
# must surface as doc_chunks LINKED TO THE PARENT doc (ai_parse_document silently ignores embedded
# objects, so the split node must extract them — a silent drop would FAIL this gate).
emb = C.rows(C.q(w, f"""
    SELECT count(*) FROM {CAT}.{SILVER}.doc_chunks
    WHERE path LIKE '%messy_with_attachment.pdf'
      AND (lower(chunk_text) LIKE '%mgmt_fee_pct%' OR lower(chunk_text) LIKE '%inv_004%'
           OR lower(chunk_text) LIKE '%fee_schedule%')"""))
emb_n = int(emb[0][0] or 0)
print(f"embedded-attachment chunks linked to parent: {emb_n} -> "
      f"{'CDI3_EMB_PASS' if emb_n > 0 else 'CDI3_EMB_FAIL'}")

In [ ]:
# §2.9 — drop scratch tables; confirm engine_perf row count
for t in (SC_PARSE, SC_CHUNK, SC_FACTS):
    C.q(w, f"DROP TABLE IF EXISTS {t}")
n_rows = int(C.rows(C.q(w, f"SELECT count(*) FROM {PERF}"))[0][0])
print(f"engine_perf rows: {n_rows}")
display(spark.table(PERF).orderBy("stage", "doc_bucket"))

## §3 — RAG quality (head-to-head)

The four retrieval arms — KA, index, supervisor, Lakebase — scored with MLflow GenAI judges + custom clause/route judges over the golden prompts, behind a corpus-parity fairness gate. One MLflow run per arm (consumed by §6's `benchmark_scores`).

In [ ]:
# §3.1 — load the golden dataset from notebooks/golden_prompts.yaml (data file, not inlined)
import yaml, pandas as pd

PROMPTS_YAML = C.repo_root() / "notebooks" / "golden_prompts.yaml"

def load_dataset(tag=None):
    raw = yaml.safe_load(PROMPTS_YAML.read_text())
    rows = []
    for p in raw["prompts"]:
        if tag and tag not in (p.get("tags") or []):
            continue
        rows.append({
            "inputs": {"query": p["inputs"]["query"]},
            "expectations": {
                "expected_facts": p["expectations"].get("expected_facts", []),
                "must_cite": p["expectations"].get("must_cite", ""),
                "guidelines": p["expectations"].get("guidelines", ""),
                "expected_route": p["expectations"].get("expected_route", ""),
            },
            "_name": p["name"], "_tags": p.get("tags", []),
        })
    return pd.DataFrame(rows)

df = load_dataset()
print(f"{len(df)} prompts; tags={sorted({t for ts in df['_tags'] for t in ts})}")

In [ ]:
# §3.2 — endpoint invocation helpers + per-arm predict_fns (each traced).
# The 3 document arms: KA (serving endpoint), the VS index queried DIRECTLY, and Lakebase
# (retained module). supervisor is its serving endpoint. Self-contained — no agents/ dependency.
import json, re, requests, mlflow

def invoke_endpoint(name, query, timeout=240):
    """POST to a serving endpoint; try the ResponsesAgent shape, then ChatAgent. Returns full JSON."""
    host = w.config.host.rstrip("/")
    url = f"{host}/serving-endpoints/{name}/invocations"
    headers = {"Authorization": w.config.authenticate()["Authorization"], "Content-Type": "application/json"}
    last = None
    for body in (
        {"input": [{"role": "user", "content": query}]},        # ResponsesAgent
        {"messages": [{"role": "user", "content": query}]},     # ChatAgent
    ):
        last = requests.post(url, headers=headers, json=body, timeout=timeout)
        if last.status_code == 200:
            return last.json()
    last.raise_for_status()

def extract_text(data):
    for item in data.get("output") or []:
        if item.get("type") == "message":
            for c in item.get("content") or []:
                if c.get("type") in ("output_text", "text"):
                    return c.get("text") or ""
        elif item.get("type") in ("output_text", "text"):
            return item.get("text") or ""
    msgs = data.get("messages") or []
    if msgs:
        return msgs[-1].get("content") or ""
    choices = data.get("choices") or []
    if choices:
        return choices[0].get("message", {}).get("content") or ""
    return json.dumps(data)[:2000]

_GENIE_SQL = re.compile(r"\bSELECT\b|\bFROM\b|\bWHERE\b|\bGROUP BY\b", re.IGNORECASE)

def detect_route(resp, text):
    """Best-effort sub-agent attribution from the MAS response trace, then answer-text heuristics."""
    blob = json.dumps(resp).lower()
    for name in ("reconciliation", "allocations", "documents_index", "documents_lakebase", "documents_ka"):
        if f'"{name}"' in blob or f"name={name}" in blob:
            return name
    if "lakebase" in blob:
        return "documents_lakebase"
    if "knowledge assistant" in blob:
        return "documents_ka"
    t = text.lower()
    if "contribution" in t or "distribution" in t or "capital account" in t:
        return "allocations"
    if _GENIE_SQL.search(text):
        return "reconciliation"
    if "[source:" in t:
        return "documents_index"
    return "unknown"

def endpoint_predict_fn(endpoint_name, with_route=False):
    @mlflow.trace(name=f"predict_{endpoint_name}")
    def predict(query: str) -> str:
        resp = invoke_endpoint(endpoint_name, query)
        text = extract_text(resp)
        return f"[route: {detect_route(resp, text)}]\n{text}" if with_route else text
    return predict

# Direct VS-index arm: query the Mosaic AI Vector Search index (hybrid) + generate with citations.
RAG_SYSTEM = ("You answer questions about fund side letters and credit agreements using ONLY the "
              "provided context chunks. Quote the relevant clause VERBATIM and cite it as "
              "[source: <file>, p.<page>]. If the answer is not in the context, say so.")

def index_predict_fn():
    from databricks.ai_search.client import AISearchClient
    from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
    vsc = AISearchClient()
    idx = vsc.get_index(endpoint_name=C.ENDPOINTS.vector_search, index_name=C.VS_INDEX)
    @mlflow.trace(name="predict_index")
    def predict(query: str) -> str:
        with mlflow.start_span(name="retrieve", span_type="RETRIEVER") as span:
            res = idx.similarity_search(query_text=query, columns=["chunk_id", "chunk_text", "path", "page"],
                                        num_results=5, query_type="HYBRID")   # VERIFY SDK (similarity_search kwargs)
            data = (res.get("result") or {}).get("data_array") or []
            hits = [{"chunk_text": r[1], "path": r[2], "page": r[3]} for r in data]
            span.set_outputs([{"page_content": h["chunk_text"],
                               "metadata": {"doc_uri": (h["path"] or "").rsplit("/", 1)[-1], "page": h["page"]}}
                              for h in hits])
        if not hits:
            return "No relevant context found in the indexed documents."
        context = "\n\n---\n\n".join(
            f"[source: {(h['path'] or '').rsplit('/',1)[-1]}, p.{h['page']}]\n{h['chunk_text']}" for h in hits)
        resp = w.serving_endpoints.query(
            name=C.ENDPOINTS.rag_chat,
            messages=[ChatMessage(role=ChatMessageRole.SYSTEM, content=RAG_SYSTEM),
                      ChatMessage(role=ChatMessageRole.USER, content=f"Context chunks:\n\n{context}\n\nQuestion: {query}")],
            temperature=0.0, max_tokens=700)
        return resp.choices[0].message.content
    return predict

# Lakebase hybrid arm (ANN + BM25 + RRF) via its retained module — retrieve then answer.
def lakebase_rag_predict_fn():
    from pipelines import lakebase_search_arm as lb
    from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
    @mlflow.trace(name="predict_lakebase_rag")
    def predict(query: str) -> str:
        with mlflow.start_span(name="retrieve", span_type="RETRIEVER") as span:
            hits = lb.retrieve(w, query, k=lb.TOP_K)
            span.set_outputs([{"page_content": h.get("chunk_text", ""),
                               "metadata": {"doc_uri": lb._short_src(h.get("path", "")), "page": h.get("page", 0)}}
                              for h in hits])
        if not hits:
            return "No relevant context found in the indexed documents."
        context = "\n\n---\n\n".join(
            f"[source: {lb._short_src(h.get('path',''))}, p.{h.get('page',0)}]\n{h.get('chunk_text','')}" for h in hits)
        resp = w.serving_endpoints.query(
            name=lb.CHAT_ENDPOINT,
            messages=[ChatMessage(role=ChatMessageRole.SYSTEM, content=lb.RAG_SYSTEM),
                      ChatMessage(role=ChatMessageRole.USER, content=f"Context chunks:\n\n{context}\n\nQuestion: {query}")],
            temperature=0.0, max_tokens=700)
        return resp.choices[0].message.content
    return predict

In [ ]:
# §3.3 — scorers: MLflow GenAI LLM judges + custom deterministic judges
from mlflow.genai.scorers import Correctness, Guidelines, RelevanceToQuery, RetrievalGroundedness, Safety, scorer

JUDGE_MODEL = os.environ.get("CDI_JUDGE_MODEL", "databricks:/databricks-claude-sonnet-4-5")

@scorer
def expected_route(outputs, expectations) -> float:
    """Did the supervisor route to the expected sub-agent? Reads the [route: <agent>] marker."""
    exp = (expectations or {}).get("expected_route", "") or ""
    if not exp:
        return 1.0
    text = outputs if isinstance(outputs, str) else str(outputs)
    m = re.search(r"\[route:\s*([a-z_]+)\]", text, re.IGNORECASE)
    return 1.0 if (m and m.group(1).lower() == exp.lower()) else 0.0

@scorer
def clause_citation(outputs, expectations) -> float:
    """Did the answer cite the expected source doc id (e.g. SL_001 / CA_010)?"""
    must = (expectations or {}).get("must_cite", "") or ""
    if not must:
        return 1.0
    text = outputs if isinstance(outputs, str) else str(outputs)
    return 1.0 if must.lower() in text.lower() else 0.0

def scorers_list(target=None):
    groundedness_guideline = Guidelines(
        name="groundedness_guideline", model=JUDGE_MODEL,
        guidelines=[
            "Every factual claim in the response must be supported by the side-letter or "
            "credit-agreement text; the response must not invent fees, rates, thresholds, "
            "dates, or clause language not present in the documents.",
            "When the question asks to quote a clause, the response must include the verbatim "
            "clause language, not a paraphrase only.",
            "Aggregate / computed numbers across many funds should be deferred to the structured "
            "Genie source rather than fabricated.",
        ])
    base = [Correctness(model=JUDGE_MODEL), RelevanceToQuery(model=JUDGE_MODEL),
            Safety(model=JUDGE_MODEL), RetrievalGroundedness(model=JUDGE_MODEL),
            clause_citation, groundedness_guideline]
    if target == "supervisor":          # only the supervisor emits a route marker
        base.append(expected_route)
    return base

In [ ]:
# §3.4 — corpus-parity fairness gate: every unstructured arm reads the SAME chunk unit (doc_chunks)
def assert_corpus_parity(targets) -> dict:
    cat, silver = C.CATALOG, C.SCHEMA_SILVER
    _base = lambda p: p.rsplit("/", 1)[-1] if p else p
    parsed_b = {_base(r[0]) for r in C.rows(C.q(w,
        f"SELECT DISTINCT coalesce(parent_path, path) FROM {cat}.{silver}.parsed "
        f"WHERE parsed IS NOT NULL AND n_elements > 0")) if r and r[0]}
    chunked_b = {_base(r[0]) for r in C.rows(C.q(w,
        f"SELECT DISTINCT path FROM {cat}.{silver}.doc_chunks")) if r and r[0]}
    missing = sorted(parsed_b - chunked_b)
    unstructured = {"ka", "index", "lakebase_rag"} & set(targets)
    parity_ok = len(chunked_b) > 0 and not missing
    print(f"[parity] parsed={len(parsed_b)} chunked={len(chunked_b)} missing={len(missing)} ok={parity_ok}")
    if unstructured and len(chunked_b) == 0:
        raise RuntimeError("CORPUS PARITY FAILED — cdi_silver.doc_chunks is EMPTY; run the SDP pipeline first.")
    if unstructured and missing:
        print(f"[parity] WARNING: {len(missing)} parsed doc(s) have no chunk (e.g. {missing[:5]}) — "
              f"unanswerable for every arm equally (still fair).")
    return {"parsed_docs": len(parsed_b), "chunked_docs": len(chunked_b), "n_missing": len(missing)}

In [ ]:
# §3.5 — run mlflow.genai.evaluate per arm; log one MLflow run each
import time

PREDICT_FOR = {
    "ka": lambda: endpoint_predict_fn(C.ENDPOINTS.knowledge_assistant),
    "index": index_predict_fn,
    "supervisor": lambda: endpoint_predict_fn(C.SUPERVISOR_ENDPOINT, with_route=True),
    "lakebase_rag": lakebase_rag_predict_fn,
}

# Serial + skip-validation: the traced retrieval predict_fns deadlock against MLflow's async
# trace-export queue under the default parallel executor.
os.environ.setdefault("MLFLOW_GENAI_EVAL_MAX_WORKERS", "1")
os.environ.setdefault("MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION", "True")

targets = ["ka", "index", "supervisor", "lakebase_rag"]
if "lakebase_rag" in targets and not C.LAKEBASE_SEARCH_ENABLED:
    print("lakebase arm disabled (set CDI_LAKEBASE_SEARCH_ENABLED=1); skipping.")
    targets.remove("lakebase_rag")

assert_corpus_parity(targets)   # fairness gate before any arm runs

exp = mlflow.get_experiment_by_name(EXPERIMENT)
eval_data = df[["inputs", "expectations"]].to_dict(orient="records")
summary = {}
for tgt in targets:
    print(f"\n{'='*60}\n-> arm={tgt}\n{'='*60}")
    predict = PREDICT_FOR[tgt]()
    with mlflow.start_run(run_name=f"rag_bench_{tgt}") as run:
        mlflow.log_param("arm", tgt)
        mlflow.log_param("n_prompts", len(df))
        result = mlflow.genai.evaluate(data=eval_data, predict_fn=predict, scorers=scorers_list(tgt))
        time.sleep(5)  # let traces flush
        summary[tgt] = {"run_id": run.info.run_id, "metrics": getattr(result, "metrics", {})}
        print(f"  run_id={run.info.run_id}\n  metrics={summary[tgt]['metrics']}")

In [ ]:
# §3.6 — latency summary per arm (read back from traces); log latency metrics onto each run
def latency_summary(run_id):
    traces = mlflow.search_traces(locations=[exp.experiment_id], run_id=run_id, max_results=200)
    if traces is None or len(traces) == 0:
        return {"n_traces": 0}
    col = next((c for c in ("execution_duration", "execution_time_ms") if c in traces.columns), None)
    if not col:
        return {"n_traces": len(traces)}
    ms = sorted(float(x) for x in traces[col].dropna().tolist())
    return {"n_traces": len(ms), "latency_p50_ms": round(ms[len(ms)//2]),
            "latency_mean_ms": round(sum(ms)/len(ms)), "latency_max_ms": round(ms[-1])}

print(f"{'='*60}\nSIDE-BY-SIDE\n{'='*60}")
for tgt, s in summary.items():
    lat = latency_summary(s["run_id"])
    for k, v in lat.items():
        if isinstance(v, (int, float)):
            with mlflow.start_run(run_id=s["run_id"]):
                mlflow.log_metric(k, v)
    print(f"\n[{tgt}] run={s['run_id']}")
    for k, v in (s["metrics"] or {}).items():
        print(f"    {k}: {v}")
    print(f"    latency: {lat}")

## §4 — Edge cases

Managed-pipeline quality vs the fixture answer keys: amendment precedence (point-in-time as-of), table-stitch continuation cells, multimodal figure (default OCR vs VLM), MFN ladder. Writes `cdi_eval.edge_case_scores`.

In [ ]:
# §4.0 — config: catalog/schemas, staging tables, parse version, extract model
CAT, SIL, EVAL = C.CATALOG, C.SCHEMA_SILVER, C.SCHEMA_EVAL
RAW = C.vol_path("docs_raw")
JUDGE = "databricks-meta-llama-3-3-70b-instruct"
PARSE_VERSION = "2.0"

T_TXT   = f"{CAT}.{SIL}._fix_parse_raw"     # text fixtures (amendment / mfn / table_stitch)
T_FIG   = f"{CAT}.{SIL}._fix_fig_raw"        # figure doc, default parse (OCR)
T_FIGV  = f"{CAT}.{SIL}._fix_fig_vlm"        # figure doc, VLM path
T_AMD   = f"{CAT}.{SIL}._fix_amd_extract"    # per-amendment baseline field extract
T_AMDC  = f"{CAT}.{SIL}._fix_amd_changes"    # per-amendment ratification-aware change facts
T_SCORES = f"{CAT}.{EVAL}.edge_case_scores"

# parsed-element accessors reused across scorers
_ELEMS    = "from_json(to_json(parsed:document:elements),'array<variant>')"
_FULLTEXT = (f"array_join(transform({_ELEMS}, "
             "x -> regexp_replace(to_json(x:content),'^\"|\"$','')), '\\n')")

In [ ]:
# §4.1 — parse the 4 fixture sets via ai_parse_document (text + figure OCR + figure VLM)
C.q(w, f"""CREATE OR REPLACE TABLE {T_TXT} AS
    SELECT regexp_replace(path,'^dbfs:','') path, ai_parse_document(content, map('version','{PARSE_VERSION}')) parsed
    FROM read_files('{RAW}/synthetic/table_stitch/', format=>'binaryFile')
    UNION ALL SELECT regexp_replace(path,'^dbfs:',''), ai_parse_document(content, map('version','{PARSE_VERSION}'))
    FROM read_files('{RAW}/synthetic/mfn_ladder/', format=>'binaryFile')
    UNION ALL SELECT regexp_replace(path,'^dbfs:',''), ai_parse_document(content, map('version','{PARSE_VERSION}'))
    FROM read_files('{RAW}/synthetic/amendment_chain/', format=>'binaryFile')""", poll_s=300)

C.q(w, f"""CREATE OR REPLACE TABLE {T_FIG} AS
    SELECT regexp_replace(path,'^dbfs:','') path, ai_parse_document(content, map('version','{PARSE_VERSION}')) parsed
    FROM read_files('{RAW}/synthetic/figure_multimodal/', format=>'binaryFile')""", poll_s=300)

C.q(w, f"""CREATE OR REPLACE TABLE {T_FIGV} AS
    SELECT regexp_replace(path,'^dbfs:','') path,
           ai_parse_document(content, map('version','{PARSE_VERSION}','descriptionElementTypes','*')) parsed
    FROM read_files('{RAW}/synthetic/figure_multimodal/', format=>'binaryFile')""", poll_s=400)
print("parsed: text fixtures, figure (OCR), figure (VLM)")

In [ ]:
# §4.2 — managed per-amendment extraction: baseline value + ratification-aware change facts
# The "no other changes" / "ratified and confirmed" trap MUST yield amends=false.
C.q(w, f"""CREATE OR REPLACE TABLE {T_AMD} AS
  WITH txt AS (SELECT regexp_extract(path,'([^/]+)\\\\.pdf$',1) doc_id, {_FULLTEXT} full_text
               FROM {T_TXT} WHERE path LIKE '%AMD_CA_%')
  SELECT doc_id, full_text, ai_query('{JUDGE}',
    CONCAT('Extract amendment metadata from ONE credit-agreement document. Return only values THIS doc explicitly states. Document:\\n', full_text),
    responseFormat => '{{"type":"json_schema","json_schema":{{"name":"a","schema":{{"type":"object","properties":{{"effective_date":{{"type":"string"}},"doc_kind":{{"type":"string","enum":["original","amendment","amended_and_restated"]}},"leverage_covenant_max":{{"type":["number","null"]}},"liquidity_covenant_min_usd":{{"type":["number","null"]}}}},"required":["effective_date","doc_kind"]}}}}}}'
  ) extracted FROM txt""", poll_s=240)

C.q(w, f"""CREATE OR REPLACE TABLE {T_AMDC} AS
  WITH txt AS (SELECT regexp_extract(path,'([^/]+)\\\\.pdf$',1) doc_id, {_FULLTEXT} full_text
               FROM {T_TXT} WHERE path LIKE '%AMD_CA_%')
  SELECT doc_id, ai_query('{JUDGE}',
    CONCAT('For the leverage_covenant_max (max Total Net Leverage Ratio): does THIS document AMEND it (a numbered Section "is amended to..." sets a NEW number, OR an Amended & Restated agreement restates it)? RATIFICATION is NOT an amendment: "remains 5.50", "for the avoidance of doubt", "all other terms remain in full force", "ratified and confirmed" => amends=false. If amends=true give new_value and change_effective_date YYYY-MM-DD (prefer a stated covenant effective date like "effective for fiscal quarters ending on or after <date>", else the document effective date). Document:\\n', full_text),
    responseFormat => '{{"type":"json_schema","json_schema":{{"name":"c","schema":{{"type":"object","properties":{{"amends_leverage":{{"type":"boolean"}},"new_leverage_value":{{"type":["number","null"]}},"change_effective_date":{{"type":["string","null"]}},"is_amended_and_restated":{{"type":"boolean"}}}},"required":["amends_leverage","is_amended_and_restated"]}}}}}}'
  ) j FROM txt""", poll_s=240)

In [ ]:
# §4.3 — scorer #6 amendment precedence: point-in-time governing leverage covenant as-of a date
def score_amendments(w):
    r = C.rows(C.q(w, f"""
      WITH base AS (SELECT doc_id, parse_json(extracted):leverage_covenant_max::double new_val,
                           to_date(parse_json(extracted):effective_date::string) chg_eff, false is_ar
                    FROM {T_AMD} WHERE parse_json(extracted):doc_kind::string='original'),
           amd AS (SELECT doc_id, parse_json(j):new_leverage_value::double new_val,
                          to_date(parse_json(j):change_effective_date::string) chg_eff,
                          parse_json(j):is_amended_and_restated::boolean is_ar
                   FROM {T_AMDC} WHERE parse_json(j):amends_leverage::boolean=true),
           cand AS (SELECT * FROM base UNION ALL SELECT * FROM amd),
           ak AS (SELECT to_date(as_of_date) as_of, CAST(expected_value AS DOUBLE) exp_val, governing_doc_id
                  FROM {CAT}.{EVAL}.answer_key_amendments WHERE field='leverage_covenant_max')
      SELECT COUNT(*) total,
        SUM(CASE WHEN
          (SELECT c.new_val FROM cand c WHERE c.chg_eff<=ak.as_of ORDER BY c.chg_eff DESC, c.is_ar DESC LIMIT 1)=ak.exp_val
          AND (SELECT c.doc_id FROM cand c WHERE c.chg_eff<=ak.as_of ORDER BY c.chg_eff DESC, c.is_ar DESC LIMIT 1)=ak.governing_doc_id
          THEN 1 ELSE 0 END) correct
      FROM ak"""))
    total, correct = int(r[0][0]), int(r[0][1] or 0)
    lr = C.rows(C.q(w, f"""
      WITH ext AS (SELECT to_date(parse_json(extracted):effective_date::string) eff,
                          parse_json(extracted):liquidity_covenant_min_usd::double liq FROM {T_AMD}),
           ak AS (SELECT to_date(as_of_date) as_of, expected_value FROM {CAT}.{EVAL}.answer_key_amendments
                  WHERE field='liquidity_covenant_min_usd')
      SELECT COUNT(*), SUM(CASE WHEN coalesce(
          (SELECT CAST(CAST(e.liq AS BIGINT) AS STRING) FROM ext e WHERE e.eff<=ak.as_of AND e.liq IS NOT NULL ORDER BY e.eff DESC LIMIT 1),
          'NONE') = ak.expected_value THEN 1 ELSE 0 END)
      FROM ak"""))
    return {"as_of_total": total, "as_of_correct": correct,
            "liquidity_total": int(lr[0][0]), "liquidity_correct": int(lr[0][1] or 0)}

In [ ]:
# §4.4 — scorer #5 table stitch: continuation-page (on_page>=2) fee-grid cell recall
def score_fee_grid(w):
    r = C.rows(C.q(w, f"""
      WITH tbl AS (SELECT to_json(e:content) html FROM (SELECT explode({_ELEMS}) e FROM {T_TXT} WHERE path LIKE '%TBL_FEE_001%')
                   WHERE to_json(e:type)='"table"'),
           rows_ex AS (SELECT explode(split(regexp_replace(html,'^"|"$',''),'<tr>')) tr FROM tbl),
           data_rows AS (SELECT tr FROM rows_ex WHERE tr LIKE '%<td>%'),
           cells AS (SELECT regexp_replace(regexp_extract(tr,'<td>(.*?)</td>',1),'<.*?>','') band,
                            filter(transform(split(tr,'<td>'), x->regexp_replace(regexp_replace(x,'</td>.*$',''),'<.*?>','')), y->y RLIKE '^[0-9]') rates
                     FROM data_rows),
           pc AS (SELECT band, CAST(REPLACE(rates[0],'%','') AS DOUBLE) r1, CAST(REPLACE(rates[1],'%','') AS DOUBLE) r2,
                         CAST(REPLACE(rates[2],'%','') AS DOUBLE) r3, CAST(REPLACE(rates[3],'%','') AS DOUBLE) r4
                  FROM cells WHERE size(rates)=4),
           unpiv AS (SELECT band,'Years 1-2' period,r1 rate FROM pc UNION ALL SELECT band,'Years 3-5',r2 FROM pc
                     UNION ALL SELECT band,'Years 6-8',r3 FROM pc UNION ALL SELECT band,'Year 9+',r4 FROM pc),
           ak AS (SELECT commitment_band, period_bucket, CAST(fee_rate_pct AS DOUBLE) rate, CAST(on_page AS INT) on_page
                  FROM {CAT}.{EVAL}.answer_key_fee_grid)
      SELECT ak.on_page, COUNT(*) total, SUM(CASE WHEN p.rate=ak.rate THEN 1 ELSE 0 END) matched
      FROM ak LEFT JOIN unpiv p ON ak.commitment_band=p.band AND ak.period_bucket=p.period
      GROUP BY ak.on_page"""))
    out = {}
    for row in r:
        out[f"page{int(row[0])}_total"] = int(row[1])
        out[f"page{int(row[0])}_matched"] = int(row[2] or 0)
    return out

In [ ]:
# §4.5 — scorer #7 multimodal figure: figure-value + stamp-date recall, default OCR vs VLM path
def score_figures(w):
    r = C.rows(C.q(w, f"""
      WITH dflt AS (SELECT array_join(transform({_ELEMS}, x->regexp_replace(to_json(x:content),'^"|"$','')),' ') t FROM {T_FIG}),
           vlm AS (SELECT array_join(transform({_ELEMS}, x->concat(regexp_replace(to_json(x:content),'^"|"$',''),' ',coalesce(regexp_replace(to_json(x:description),'^"|"$',''),''))),' ') t FROM {T_FIGV}),
           ak AS (SELECT modality, CASE WHEN modality='figure' THEN CONCAT('$',FORMAT_NUMBER(ABS(CAST(expected_value AS BIGINT)),0))
                                        WHEN modality='stamp' THEN '2026-03-31' ELSE expected_value END needle
                  FROM {CAT}.{EVAL}.answer_key_figures)
      SELECT ak.modality, COUNT(*) total,
        SUM(CASE WHEN (SELECT t FROM dflt) LIKE CONCAT('%',ak.needle,'%') THEN 1 ELSE 0 END) found_default,
        SUM(CASE WHEN (SELECT t FROM vlm)  LIKE CONCAT('%',ak.needle,'%') THEN 1 ELSE 0 END) found_vlm
      FROM ak GROUP BY ak.modality"""))
    out = {}
    for row in r:
        m = row[0]
        out[f"{m}_total"] = int(row[1]); out[f"{m}_default"] = int(row[2] or 0); out[f"{m}_vlm"] = int(row[3] or 0)
    return out

In [ ]:
# §4.6 — scorer MFN ladder: resolved entitlement ($150MM LP -> Tier 2 -> economic_terms) + window + excluded
def score_mfn(w):
    j = C.rows(C.q(w, f"""
      WITH txt AS (SELECT {_FULLTEXT} t FROM {T_TXT} WHERE path LIKE '%MFN_SL_001%')
      SELECT ai_query('{JUDGE}',
        CONCAT('Extract the tiered MFN election. Resolve the SUBJECT investor entitlement (tier + scope) from their commitment. Map scope to: any_term, economic_terms, fee_terms, none. List election window days and excluded investors. Document:\\n', t),
        responseFormat => '{{"type":"json_schema","json_schema":{{"name":"m","schema":{{"type":"object","properties":{{"resolved_tier":{{"type":"string"}},"resolved_scope":{{"type":"string","enum":["any_term","economic_terms","fee_terms","none"]}},"election_window_days":{{"type":"integer"}},"excluded_investors":{{"type":"array","items":{{"type":"string"}}}}}},"required":["resolved_tier","resolved_scope","election_window_days","excluded_investors"]}}}}}}'
      ) j FROM txt"""))
    ext = json.loads(j[0][0])
    ak = C.rows(C.q(w, f"""SELECT tier_label, electable_scope FROM {CAT}.{EVAL}.answer_key_mfn_ladder
                           WHERE row_type='resolved_entitlement'"""))
    exp_tier, exp_scope = ak[0][0], ak[0][1]
    excl = C.rows(C.q(w, f"""SELECT electable_scope FROM {CAT}.{EVAL}.answer_key_mfn_ladder WHERE row_type='excluded_investor'"""))
    exp_excl = {row[0] for row in excl}
    got_excl = set(ext.get("excluded_investors", []))
    return {"tier_correct": int(ext.get("resolved_tier") == exp_tier),
            "scope_correct": int(ext.get("resolved_scope") == exp_scope),
            "window_correct": int(ext.get("election_window_days") == 30),
            "excluded_recall_n": len(exp_excl & got_excl), "excluded_total": len(exp_excl)}

In [ ]:
# §4.7 — run all four scorers
import json
results = {"amendments": score_amendments(w), "fee_grid": score_fee_grid(w),
           "figures": score_figures(w), "mfn": score_mfn(w)}
print(json.dumps(results, indent=2))

In [ ]:
# §4.8 — persist scored metric rows to cdi_eval.edge_case_scores
C.q(w, f"""CREATE TABLE IF NOT EXISTS {T_SCORES} (
    edge_case STRING, metric STRING, value DOUBLE, detail STRING, scored_at TIMESTAMP) USING DELTA""")

a, f, g, m = results["amendments"], results["fee_grid"], results["figures"], results["mfn"]
score_rows = [
    ("#6_amendment_precedence", "as_of_leverage_accuracy",
     a["as_of_correct"]/a["as_of_total"], f"{a['as_of_correct']}/{a['as_of_total']} point-in-time leverage as-of dates"),
    ("#6_amendment_precedence", "liquidity_honesty_accuracy",
     a["liquidity_correct"]/a["liquidity_total"], f"{a['liquidity_correct']}/{a['liquidity_total']} (incl. NONE no-fabrication row)"),
    ("#5_table_stitch", "continuation_cell_recall",
     f.get("page2_matched",0)/max(f.get("page2_total",1),1), f"on_page>=2: {f.get('page2_matched',0)}/{f.get('page2_total',0)} cells"),
    ("#5_table_stitch", "page1_cell_recall",
     f.get("page1_matched",0)/max(f.get("page1_total",1),1), f"on_page=1: {f.get('page1_matched',0)}/{f.get('page1_total',0)} cells"),
    ("#7_multimodal_figure", "figure_recall_default_ocr",
     g.get("figure_default",0)/max(g.get("figure_total",1),1), f"{g.get('figure_default',0)}/{g.get('figure_total',0)} figure values (default OCR)"),
    ("#7_multimodal_figure", "figure_recall_vlm",
     g.get("figure_vlm",0)/max(g.get("figure_total",1),1), f"{g.get('figure_vlm',0)}/{g.get('figure_total',0)} figure values (descriptionElementTypes='*')"),
    ("#7_multimodal_figure", "stamp_date_recall_default_ocr",
     g.get("stamp_default",0)/max(g.get("stamp_total",1),1), "stamp date via default OCR (truncated -> miss)"),
    ("#7_multimodal_figure", "stamp_date_recall_vlm",
     g.get("stamp_vlm",0)/max(g.get("stamp_total",1),1), "stamp date via VLM path (recovered)"),
    ("mfn_ladder", "resolved_entitlement_scope",
     float(m["scope_correct"]), f"$150MM LP -> Tier 2 -> economic_terms; tier_correct={m['tier_correct']}"),
    ("mfn_ladder", "election_window_correct", float(m["window_correct"]), "30-day window"),
    ("mfn_ladder", "excluded_investor_recall",
     m["excluded_recall_n"]/max(m["excluded_total"],1), f"{m['excluded_recall_n']}/{m['excluded_total']} excluded investors"),
]
vals = ", ".join(f"('{ec}','{me}',{v},'{d.replace(chr(39),chr(39)*2)}',current_timestamp())" for (ec,me,v,d) in score_rows)
C.q(w, f"INSERT INTO {T_SCORES} (edge_case, metric, value, detail, scored_at) VALUES {vals}")
display(C.q(w, f"SELECT edge_case, metric, value, detail FROM {T_SCORES} ORDER BY scored_at DESC, edge_case, metric"))

## §5 — Cross-cutting QA golden set

Materialize the curated hybrid (Genie + RAG) question set into `cdi_eval.cross_cutting_qa`; the `is_suggested` rows also feed the app's empty-state suggestions.

In [ ]:
# §5.0 — load curated cross-cutting prompts from the data file (do NOT inline the prompts)
import yaml
from databricks.sdk.service.sql import StatementParameterListItem

CC_TABLE = f"{CAT}.{EVAL}.cross_cutting_qa"
YAML_PATH = C.repo_root() / "notebooks" / "cross_cutting_prompts.yaml"

prompts = yaml.safe_load(YAML_PATH.read_text())["prompts"]
print(f"loaded {len(prompts)} prompts "
      f"({sum(1 for p in prompts if p.get('answerable'))} answerable, "
      f"{sum(1 for p in prompts if p.get('is_suggested'))} suggested)")

In [ ]:
# §5.1 — (re)create cdi_eval.cross_cutting_qa
# Feeds the eval golden set AND the app empty-state suggested questions (is_suggested=true).
C.q(w, f"""CREATE OR REPLACE TABLE {CC_TABLE} (
    id STRING, question STRING, category STRING,
    requires_rag BOOLEAN, requires_genie_spaces ARRAY<STRING>, spans_both_spaces BOOLEAN,
    expected_answer STRING, supporting_ids STRING,
    is_suggested BOOLEAN, answerable BOOLEAN, evidence STRING, created_at TIMESTAMP
) USING DELTA""")

In [ ]:
# §5.2 — insert the answerable rows (parameterized; curated/doc-derived text never interpolated)
def insert_cc(w, p):
    spaces = p.get("requires_genie_spaces") or []
    array_expr = f"array({', '.join(f':s{i}' for i in range(len(spaces)))})" if spaces else "array()"
    params = {
        "id": p["id"], "question": " ".join(str(p["question"]).split()), "category": p.get("category", ""),
        "requires_rag": bool(p.get("requires_rag", False)), "spans_both_spaces": bool(p.get("spans_both_spaces", False)),
        "expected_answer": " ".join(str(p.get("expected_answer", "")).split()), "supporting_ids": str(p.get("supporting_ids", "")),
        "is_suggested": bool(p.get("is_suggested", False)), "answerable": True,
        "evidence": " ".join(str(p.get("evidence", "")).split()),
    }
    for i, s in enumerate(spaces):
        params[f"s{i}"] = s
    sql = f"""INSERT INTO {CC_TABLE} (
        id, question, category, requires_rag, requires_genie_spaces, spans_both_spaces,
        expected_answer, supporting_ids, is_suggested, answerable, evidence, created_at
      ) VALUES (
        :id, :question, :category, :requires_rag, {array_expr}, :spans_both_spaces,
        :expected_answer, :supporting_ids, :is_suggested, :answerable, :evidence, current_timestamp())"""
    plist = [StatementParameterListItem(
                name=k, value=("true" if v else "false") if isinstance(v, bool) else str(v))
             for k, v in params.items()]
    r = w.statement_execution.execute_statement(
        warehouse_id=C.WAREHOUSE_ID, statement=sql, parameters=plist, wait_timeout="30s")
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(f"INSERT failed for {p['id']}: {r.status.error.message}")

inserted = 0
for p in prompts:
    if p.get("answerable"):
        insert_cc(w, p); inserted += 1
print(f"inserted {inserted} answerable rows into {CC_TABLE}")
display(C.q(w, f"""SELECT id, category, requires_rag, spans_both_spaces, is_suggested, supporting_ids
                   FROM {CC_TABLE} ORDER BY id"""))

_Optional live re-validation (not run): the `answerable` flags were hand-vetted against live data. To re-probe drift, ask each row's Genie space (`w.genie.start_conversation_and_wait`) and query the VS index (`w.vector_search_indexes.query_index`, HYBRID) — a check, never a mutation. Skipped here to keep §5 fast and deterministic._

## §6 — Summary rollup

Write `cdi_eval.benchmark_scores` (latest MLflow run per arm) and `cdi_eval.pipeline_metrics` (live bronze/silver/gold counts) — the tables behind the app's insights tab.

In [ ]:
# §6.0 — config: MLflow benchmark experiment (workspace store, no local sqlite)
import pandas as pd
# EXPERIMENT + mlflow were configured in §1; the §3 RAG benchmark wrote one run per arm.

In [ ]:
# §6.1 — benchmark_scores: latest MLflow run per arm -> long {arm, metric, value} rows
def fetch_mlflow_scores():
    exp = mlflow.get_experiment_by_name(EXPERIMENT)
    if exp is None:
        raise RuntimeError(f"MLflow experiment {EXPERIMENT} not found — re-run the §3 RAG benchmark.")
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], max_results=200, order_by=["start_time DESC"])
    runs = runs[runs["params.arm"].notna()]
    if runs.empty:
        raise RuntimeError("no benchmark runs with params.arm found — run §3 first")
    latest = runs.sort_values("start_time").groupby("params.arm").tail(1)   # newest run per arm
    metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
    rows = []
    for _, r in latest.iterrows():
        for mc in metric_cols:
            v = r[mc]
            if pd.notna(v):
                rows.append({"arm": r["params.arm"], "metric": mc.replace("metrics.", ""),
                             "value": round(float(v), 4),
                             "n_prompts": int(r["params.n_prompts"]) if pd.notna(r.get("params.n_prompts")) else None,
                             "run_id": r["run_id"]})
    return rows

def write_benchmark_scores(w, rows):
    C.q(w, f"DROP TABLE IF EXISTS {EVAL}.benchmark_scores")
    C.q(w, f"""CREATE TABLE {EVAL}.benchmark_scores (
            arm STRING, metric STRING, value DOUBLE, n_prompts INT, run_id STRING
        ) COMMENT 'Per-arm MLflow GenAI judge scores + latency (KA vs index vs supervisor vs lakebase). From /Shared/citco-cdi/rag-benchmark.'""")
    vals = []
    for r in rows:
        npv = "NULL" if r["n_prompts"] is None else str(r["n_prompts"])
        vals.append(f"('{r['arm']}','{r['metric']}',{r['value']},{npv},'{r['run_id']}')")
    C.q(w, f"INSERT INTO {EVAL}.benchmark_scores VALUES " + ",".join(vals))
    return len(rows)

scores = fetch_mlflow_scores()
n_bench = write_benchmark_scores(w, scores)
print(f"benchmark_scores: {n_bench} rows, arms={sorted({r['arm'] for r in scores})}")

In [ ]:
# §6.2 — pipeline_metrics: doc/parse/extract counts read live from bronze/silver/gold + measured timings
def write_pipeline_metrics(w):
    C.q(w, f"DROP TABLE IF EXISTS {EVAL}.pipeline_metrics")
    C.q(w, f"""CREATE TABLE {EVAL}.pipeline_metrics (
            category STRING, metric STRING, value DOUBLE, unit STRING, sort_order INT
        ) COMMENT 'Doc/parse/extract creation metrics. Counts read live from bronze/silver/gold; timings are measured wall clocks.'""")
    extract_secs = 149   # measured set-based extract wall clock
    C.q(w, f"""
      INSERT INTO {EVAL}.pipeline_metrics
      SELECT 'Corpus','Total docs ingested', CAST(count(*) AS DOUBLE),'docs',10 FROM {C.CATALOG}.{C.SCHEMA_BRONZE}.corpus_manifest
      UNION ALL SELECT 'Corpus','Real (EDGAR/ILPA)', CAST(count(*) AS DOUBLE),'docs',11 FROM {C.CATALOG}.{C.SCHEMA_BRONZE}.corpus_manifest WHERE source='real'
      UNION ALL SELECT 'Corpus','Synthetic (answer-key)', CAST(count(*) AS DOUBLE),'docs',12 FROM {C.CATALOG}.{C.SCHEMA_BRONZE}.corpus_manifest WHERE source='synthetic'
      UNION ALL SELECT 'Corpus','Messy (edge cases)', CAST(count(*) AS DOUBLE),'docs',13 FROM {C.CATALOG}.{C.SCHEMA_BRONZE}.corpus_manifest WHERE source='messy'
      UNION ALL SELECT 'Parsing','Parsed OK', CAST(count(distinct coalesce(parent_path, path)) AS DOUBLE),'docs',20 FROM {C.CATALOG}.{C.SCHEMA_SILVER}.parsed WHERE parsed IS NOT NULL AND n_elements > 0
      UNION ALL SELECT 'Parsing','Dead-letter (honest fail)', CAST(count(*) AS DOUBLE),'docs',21 FROM {C.CATALOG}.{C.SCHEMA_SILVER}._dead_letter
      UNION ALL SELECT 'Parsing','Parsed elements', CAST(coalesce(sum(n_elements),0) AS DOUBLE),'elements',23 FROM {C.CATALOG}.{C.SCHEMA_SILVER}.parsed
      UNION ALL SELECT 'RAG','Chunks indexed', CAST(count(*) AS DOUBLE),'chunks',30 FROM {C.CATALOG}.{C.SCHEMA_SILVER}.doc_chunks
      UNION ALL SELECT 'Extraction','Side-letter terms', CAST(count(*) AS DOUBLE),'rows',40 FROM {C.CATALOG}.{C.SCHEMA_GOLD}.side_letter_terms
      UNION ALL SELECT 'Extraction','Credit-agreement terms', CAST(count(*) AS DOUBLE),'rows',41 FROM {C.CATALOG}.{C.SCHEMA_GOLD}.credit_agreement_terms
      UNION ALL SELECT 'Timing','Extract wall clock (set-based)', CAST({extract_secs} AS DOUBLE),'seconds',50
    """)
    return int(C.rows(C.q(w, f"SELECT count(*) FROM {EVAL}.pipeline_metrics"))[0][0])

n_pipe = write_pipeline_metrics(w)
print(f"pipeline_metrics: {n_pipe} rows")

In [ ]:
# §6.3 — summary of the app insights tables
print(f"DONE -> {EVAL}: benchmark_scores ({n_bench}), pipeline_metrics ({n_pipe})")